Document Structure

In [1]:
from langchain_core.documents import Document
from langchain_text_splitters  import RecursiveCharacterTextSplitter

c:\Users\Asus\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
os.makedirs("../data/text_files", exist_ok=True)

In [3]:
sample_texts = {
    "../data/text_files/sample1.txt": "This is the content of sample text file 1.",
}

for file_path, content in sample_texts.items():
    with open(file_path, "w") as f:
        f.write(content)

print("Sample text files created successfully.")

Sample text files created successfully.


Text Loader

In [4]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/sample1.txt")
doc = loader.load()

print(doc)

[Document(metadata={'source': '../data/text_files/sample1.txt'}, page_content='This is the content of sample text file 1.')]


C:\Users\Asus\AppData\Local\Temp\ipykernel_4948\1602887145.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


Directory Loader

In [31]:
from langchain_community.document_loaders import DirectoryLoader

loader = DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=False
)
docs = loader.load()

print(docs)

[Document(metadata={'source': '..\\data\\text_files\\sample1.txt'}, page_content='This is the content of sample text file 1.'), Document(metadata={'source': '..\\data\\text_files\\sample2.txt'}, page_content='Artificial intelligence is transforming how we create, learn, and solve problems.\nBy analyzing data, recognizing patterns, and adapting to new situations, AI systems can help people make smarter decisions and build new kinds of applications.\nAs AI evolves, it opens possibilities for better tools, more efficient workflows, and a deeper understanding of how technology supports human creativity.')]


In [32]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader

loader = DirectoryLoader(
    "../data",
    glob="**/*.pdf",
    loader_cls=PyPDFLoader,
)
docs = loader.load()

print(docs)

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-02T15:25:28+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-02T15:25:28+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\text_files\\About_ChatGPT_500_Words.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='About ChatGPT\nI am ChatGPT, an artificial intelligence language model created by OpenAI. My purpose is to assist\npeople by answering questions, explaining concepts, generating ideas, helping with writing, solving\nprogramming problems, and supporting learning across a wide range of subjects. Rather than\nthinking or feeling as humans do, I generate responses by recognizing patterns in language learned\nduring training and by using the context provided in each conversation. One of my strengths is\nversatility. I can help students understand mathematics, science, liter

In [33]:
def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

chunks = split_documents(docs)
chunks

Split 1 documents into 4 chunks

Example chunk:
Content: About ChatGPT
I am ChatGPT, an artificial intelligence language model created by OpenAI. My purpose is to assist
people by answering questions, explaining concepts, generating ideas, helping with writ...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-02T15:25:28+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-02T15:25:28+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\text_files\\About_ChatGPT_500_Words.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-08-02T15:25:28+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-08-02T15:25:28+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\text_files\\About_ChatGPT_500_Words.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='About ChatGPT\nI am ChatGPT, an artificial intelligence language model created by OpenAI. My purpose is to assist\npeople by answering questions, explaining concepts, generating ideas, helping with writing, solving\nprogramming problems, and supporting learning across a wide range of subjects. Rather than\nthinking or feeling as humans do, I generate responses by recognizing patterns in language learned\nduring training and by using the context provided in each conversation. One of my strengths is\nversatility. I can help students understand mathematics, science, liter

Embeddings And VectorDB

In [34]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [35]:
class EmbeddingManager:
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading model {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model {self.model_name} loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
            if not self.model:
                raise ValueError("Model is not loaded.")

            # 1. Force into a list if a single string or single object is passed
            if isinstance(texts, str) or hasattr(texts, 'page_content') or hasattr(texts, 'text'):
                texts = [texts]
            
            clean_texts = []
            
            # 2. Flatten and parse the inputs
            def extract_text(item):
                if item is None:
                    return None
                # Handle standard LangChain Document
                if hasattr(item, 'page_content'):
                    return item.page_content
                # Handle LlamaIndex Document/Node
                if hasattr(item, 'text'):
                    return item.text
                # Handle dictionaries with common text keys
                if isinstance(item, dict):
                    return item.get('text', item.get('page_content', str(item)))
                return str(item)

            # Iterate and recursively flatten if a list of lists was accidentally passed
            for t in texts:
                if isinstance(t, (list, tuple, np.ndarray)):
                    for sub_t in t:
                        text_val = extract_text(sub_t)
                        if text_val and text_val.strip():
                            clean_texts.append(text_val.strip())
                else:
                    text_val = extract_text(t)
                    if text_val and text_val.strip():
                        clean_texts.append(text_val.strip())

            # 3. Check for empty text arrays
            if not clean_texts:
                try:
                    dim = self.model.get_embedding_dimension()
                except Exception:
                    return np.empty((0,))
                return np.empty((0, dim))

            print(f"Generating embeddings for {len(clean_texts)} texts...")
            
            # 4. Batch encoding
            batch_size = 64
            embeddings_batches = []
            for i in range(0, len(clean_texts), batch_size):
                batch = clean_texts[i:i+batch_size]
                # convert directly to numpy array to avoid overhead
                emb = self.model.encode(batch, show_progress_bar=False, convert_to_numpy=True)
                embeddings_batches.append(emb)
                
            embeddings = np.vstack(embeddings_batches)
            return embeddings


Initialize the Embedding Manager

In [36]:
embedding_manager = EmbeddingManager()

Loading model all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3526.06it/s]


Model all-MiniLM-L6-v2 loaded successfully. Embedding dimension: 384


Vector Store

In [37]:
class VectorStore:
    def __init__(self, collection_name: str = "document_embeddings", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(
                path=self.persist_directory,
                settings=Settings(allow_reset=True),
            )
            

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name, 
                metadata={
                    "description": "Collection of document embeddings",
                    "hnsw:space": "cosine"
                }
            )

            print(f"ChromaDB collection '{self.collection_name}' initialized successfully.")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing ChromaDB client: {e}")
            raise

    def add_documents(self, documents: List[Document], embeddings: np.ndarray):
        if len(documents) != embeddings.shape[0]:
            raise ValueError("Number of documents and embeddings must match.")
        
        if not documents:
            print("No documents to add.")
            return
        
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"error adding documents to vector store: {e}")
            raise

        print(f"Added {len(documents)} documents to the collection '{self.collection_name}'.")  


vector_store = VectorStore()
vector_store

ChromaDB collection 'document_embeddings' initialized successfully.
Existing documents in collection: 0


Convert Text to Embeddings

In [38]:
texts = [doc.page_content for doc in chunks]
embeddings = embedding_manager.generate_embeddings(texts)

Generating embeddings for 4 texts...


Store the embeddings in VectorDB

In [39]:
vector_store.add_documents(chunks, embeddings)

total documents in collection: 4
Added 4 documents to the collection 'document_embeddings'.


Retriever Pipeline From VectorStore

In [44]:
class RAGRetriever:
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieval(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        print(f"Retrieving documents for query: {query}")
        print(f"Top K = {top_k}, Score Threshold = {score_threshold}")

        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings = [query_embedding.tolist()],
                n_results = top_k
            )

            retrieved_docs = []
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print(f"No documents found")

            return retrieved_docs
        except Exception as e:
            print("No document retrieved")
            return []
rag_retriever = RAGRetriever(vector_store, embedding_manager)

rag_retriever

In [41]:
rag_retriever.retrieval("Who created ChatGPT?")

Retrieving documents for query: Who created ChatGPT?
Top K = 5, Score Threshold = 0.0
Generating embeddings for 1 texts...
{'hnsw:space': 'cosine', 'description': 'Collection of document embeddings'}
Retrieved 4 documents (after filtering)


[{'id': 'doc_1ffc1207_0',
  'content': 'About ChatGPT\nI am ChatGPT, an artificial intelligence language model created by OpenAI. My purpose is to assist\npeople by answering questions, explaining concepts, generating ideas, helping with writing, solving\nprogramming problems, and supporting learning across a wide range of subjects. Rather than\nthinking or feeling as humans do, I generate responses by recognizing patterns in language learned\nduring training and by using the context provided in each conversation. One of my strengths is\nversatility. I can help students understand mathematics, science, literature, history, and computer\nscience. I can assist software developers with debugging code, explaining algorithms, and\nsuggesting better approaches to problem-solving. I also help professionals draft emails, reports,\npresentations, and other documents. Creative tasks are another area where I can contribute,\nincluding brainstorming stories, poems, scripts, or marketing ideas. Alt

Integration VectorDB Context Pipeline with LLM output

In [52]:
from google import genai
import os
from dotenv import load_dotenv
load_dotenv()

gemini_api_key = os.getenv("GEMINI_API_KEY")

client = genai.Client(api_key=gemini_api_key)

def simple_rag(query, retriever, llm, top_k=3):
    retrieved_docs = retriever.retrieval(query, top_k=top_k)
    
    if not retrieved_docs:
        print("No relevant documents found.")
        return None
    
    context = "\n\n".join([f"Document {doc['rank']} (Score: {doc['similarity_score']:.4f}):\n{doc['content']}" for doc in retrieved_docs])
    
    prompt = f"""You are a helpful assistant.

            Use ONLY the provided context to answer the question.
            If the answer is not present in the context, say:
            "I couldn't find that information in the provided documents."

            Context:
            {context}

            Question:
            {query}

            Answer:
            
            """
    
    response = client.interactions.create(
        model="gemini-3.1-flash-lite",
        input=prompt
    )
        
    return response.output_text

In [53]:
ans = simple_rag("Who created ChatGPT?", rag_retriever, client, top_k=3)


Retrieving documents for query: Who created ChatGPT?
Top K = 3, Score Threshold = 0.0
Generating embeddings for 1 texts...
Retrieved 3 documents (after filtering)


In [54]:
print(ans)

ChatGPT was created by OpenAI.


Enhanced Pipeline Features

In [56]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features --
        returns answer, sources, confidence score, and optionally full context
    """

    results = retriever.retrieval(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {
            'answer': 'No relevant context found',
            'sources': [],
            'confidence':0.0,
            'context': ''
        }

    context = "\n\n".join(doc['content'] for doc in results)
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:220] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    prompt = f"""You are a helpful assistant.
    
                Use ONLY the provided context to answer the question.
                If the answer is not present in the context, say:
                "I couldn't find that information in the provided documents."
    
                Context:
                {context}
    
                Question:
                {query}
    
                Answer:
                
                """
    response = client.interactions.create(
        model="gemini-3.1-flash-lite",
        input=prompt
    )
    output = {
        'answer': response.output_text,
        'sources': sources,
        'confidence': confidence,
    }
    if return_context:
        output['context'] = context

    return output
    

In [ ]:
# Example usage of the advanced RAG function
result = rag_advanced("What is the purpose of ChatGPT?", rag_retriever, client, top_k=3, min_score=0.2, return_context=True)
print("Answer", result['answer'])
print("Sources", result['sources'])
print("Confidence", result['confidence'])
print("Context", result['context'][:300])

Retrieving documents for query: What is the purpose of ChatGPT?
Top K = 3, Score Threshold = 0.2
Generating embeddings for 1 texts...
Retrieved 2 documents (after filtering)
Answer The purpose of ChatGPT is to assist people by answering questions, explaining concepts, generating ideas, helping with writing, solving programming problems, and supporting learning across a wide range of subjects. Ultimately, its goal is to make information more accessible and to help people think, learn, create, and solve problems more effectively.
Sources [{'source': '..\\data\\text_files\\About_ChatGPT_500_Words.pdf', 'page': 0, 'score': 0.43681520223617554, 'preview': 'About ChatGPT\nI am ChatGPT, an artificial intelligence language model created by OpenAI. My purpose is to assist\npeople by answering questions, explaining concepts, generating ideas, helping with writing, solving\nprogram...'}, {'source': '..\\data\\text_files\\About_ChatGPT_500_Words.pdf', 'page': 0, 'score': 0.34999728202819824, 'prev